# A Thousand Questions at Once — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 10: A Thousand Questions at Once](https://learnai.robennals.org/matrices). You'll build a matrix as a stack of dot products, confirm that `nn.Linear` is a matrix multiply, watch attention score keys against a query, and check for yourself that matrix × matrix is just one matrix.

*New to PyTorch? Start with the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a quick intro to tensors.*

In [ ]:
import torch
import torch.nn as nn

## Our Toy Animals

We'll use the same animals as the chapter. Each animal is rated 0–1 on six properties:
**big**, **scary**, **hairy**, **cuddly**, **fast**, **fat**.

Each animal is a **vector** — just a list of six numbers.

In [ ]:
animals = {
    'Bear':     torch.tensor([0.90, 0.85, 0.80, 0.50, 0.40, 0.75]),
    'Rabbit':   torch.tensor([0.10, 0.02, 0.60, 0.95, 0.70, 0.15]),
    'Shark':    torch.tensor([0.80, 0.95, 0.00, 0.00, 0.75, 0.20]),
    'Mouse':    torch.tensor([0.02, 0.05, 0.30, 0.40, 0.60, 0.10]),
    'Eagle':    torch.tensor([0.35, 0.60, 0.05, 0.02, 0.95, 0.05]),
    'Elephant': torch.tensor([0.98, 0.30, 0.05, 0.40, 0.15, 0.95]),
    'Snake':    torch.tensor([0.20, 0.85, 0.00, 0.02, 0.50, 0.05]),
    'Cat':      torch.tensor([0.15, 0.30, 0.75, 0.85, 0.70, 0.25]),
    'Dog':      torch.tensor([0.45, 0.20, 0.70, 0.90, 0.55, 0.45]),
}

# Properties are: big, scary, hairy, cuddly, fast, fat
print(animals['Bear'])


## A Matrix Is Many Dot Products at Once

The chapter built a **bear detector**: take an animal vector, dot it with a "bear" weight-vector, and out comes a single number — how bear-like this animal is.

Let's verify that first.

In [ ]:
bear = animals['Bear']
dog  = animals['Dog']

# The bear detector: dot(bear_weights, dog)
score = torch.dot(bear, dog)
print(f"Dog's bear-likeness score: {score:.3f}")


Now suppose you want a **bear** detector, an **eagle** detector, and a **snake** detector all at once.

Stack their weight-vectors as the **rows** of a matrix `W`. Multiplying any animal by `W` runs every detector in a single shot: the animal comes out re-described as a vector of match scores.

In [ ]:
# Stack three reference animals as rows of a weight matrix
reference_names = ['Bear', 'Eagle', 'Snake']
W = torch.stack([animals[name] for name in reference_names])
print(f"Weight matrix W shape: {W.shape}  (3 detectors × 6 properties)")
print(f"Row 0 = Bear weights:  {W[0]}")
print(f"Row 1 = Eagle weights: {W[1]}")
print(f"Row 2 = Snake weights: {W[2]}")


In [ ]:
dog = animals['Dog']

# Matrix-vector multiply: all three dot products at once
scores_matmul = W @ dog

# Verify: this equals stacking the individual dot products
scores_manual = torch.tensor([
    torch.dot(animals['Bear'],  dog).item(),
    torch.dot(animals['Eagle'], dog).item(),
    torch.dot(animals['Snake'], dog).item(),
])

print("Dog re-described as match scores:")
for name, s in zip(reference_names, scores_matmul):
    print(f"  {name:6s}: {s:.3f}")

print(f"\nW @ dog equals stacked dot products? {torch.allclose(scores_matmul, scores_manual)}")


That's all a matrix multiply is. Each row asks its own question (a dot product), and the output vector collects every answer at once.

## Can You Get the Original Back?

Re-describing Dog as "how bear-like, how eagle-like, how snake-like" replaces its original six properties with three match scores. Can we recover the original?

With a **square** matrix (same number of detectors as properties) built from six varied reference animals, the transformation is lossless. The **inverse** of `W` undoes it exactly.

In [ ]:
# Build a square 6×6 weight matrix from six distinct reference animals
six_names = ['Bear', 'Eagle', 'Snake', 'Rabbit', 'Elephant', 'Cat']
W6 = torch.stack([animals[name] for name in six_names])
print(f"Square W6 shape: {W6.shape}")

dog = animals['Dog']

# Re-describe Dog
compressed = W6 @ dog

# Recover the original using the inverse
W6_inv = torch.linalg.inv(W6)
recovered = W6_inv @ compressed

print(f"\nOriginal Dog:  {dog}")
print(f"Recovered Dog: {recovered}")
print(f"Max abs difference: {(dog - recovered).abs().max():.2e}  (should be ≈ 0)")


Now use **fewer** detectors than properties — squeeze six properties into three numbers. Three numbers can't encode six, so the compression is lossy and there's no inverse.

In [ ]:
# Non-square 3×6 matrix: three detectors, six properties
W3 = torch.stack([animals['Bear'], animals['Eagle'], animals['Snake']])
print(f"Non-square W3 shape: {W3.shape}  (3 outputs from 6 inputs)")

lossy = W3 @ dog
print(f"\nDog compressed to 3 numbers: {lossy}")

# torch.linalg.inv requires a square matrix — verify this fails
try:
    torch.linalg.inv(W3)
except RuntimeError as e:
    print(f"\nCan't invert a non-square matrix: {e}")

print("\nThree numbers can't recover six — the detail is gone.")
print("(This lossy squeeze is what neural-network embedding layers do: see the Embeddings chapter.)")


## A Layer Is `nn.Linear` + an Activation

A neural-network layer applies a matrix multiply and then an **activation function** — a "bend" that zeroes out weak matches.

Let's first verify that `nn.Linear` really is just a matrix multiply.

In [ ]:
torch.manual_seed(0)

# A linear layer: 6 inputs → 4 outputs, no bias for simplicity
layer = nn.Linear(6, 4, bias=False)

W = layer.weight  # the weight matrix, shape (4, 6)
print(f"layer.weight shape: {W.shape}  (4 detectors × 6 properties)")

dog = animals['Dog']

# Through the layer
layer_out = layer(dog)

# By hand: W @ x
manual_out = W @ dog

print(f"\nnn.Linear output: {layer_out}")
print(f"W @ dog:          {manual_out}")
print(f"\nIdentical? {torch.allclose(layer_out, manual_out)}")


Now add an activation. `torch.relu` zeroes out every negative value — weak matches (below-zero scores) are silenced, strong matches survive. **A layer = matrix multiply + a bend.**

(ReLU and other activations are covered in detail in the Training chapter.)

In [ ]:
scores = W @ dog
print("Scores before ReLU:", scores.detach())

activated = torch.relu(scores)
print("Scores after  ReLU:", activated.detach())
print("\nNegative (weak) matches are zeroed out; positive (strong) matches pass through.")


## Matrix × Matrix Is Just One Matrix

Stack two linear layers with **no activation** between them and they collapse into one.
We can multiply the two weight matrices together first and get a single matrix that does both steps at once.

In [ ]:
# Use matrices with some negative intermediate values so ReLU has an effect
A = torch.tensor([[ 1.0, -1.5],
                  [ 0.5,  0.8]])

B = torch.tensor([[ 0.6,  0.3],
                  [-0.4,  1.2]])

x = torch.tensor([1.0, 2.0])

# Apply A then B (no activation between them)
two_layers = B @ (A @ x)

# Pre-multiply the matrices: BA is a single 2×2 matrix
C = B @ A
one_layer = C @ x

print(f"A @ x = {(A @ x).tolist()}  (intermediate values)")
print(f"B @ (A @ x):  {two_layers.tolist()}")
print(f"(B @ A) @ x:  {one_layer.tolist()}")
print(f"Same result? {torch.allclose(two_layers, one_layer)}")
print("\n→ Two matrix layers without activation collapse into one. Depth buys nothing.")


In [ ]:
# With a nonlinear activation between them they can NO LONGER be merged.
# ReLU clips the negative intermediate value, breaking the linear chain.
intermediate = A @ x
print(f"Intermediate before ReLU: {intermediate.tolist()}")
print(f"Intermediate after  ReLU: {torch.relu(intermediate).tolist()}  (negative zeroed out)")

two_layers_relu = B @ torch.relu(intermediate)
one_layer_result = C @ x

print(f"\nB @ relu(A @ x):  {two_layers_relu.tolist()}")
print(f"(B @ A) @ x:      {one_layer_result.tolist()}")
print(f"Same result? {torch.allclose(two_layers_relu, one_layer_result)}")
print("\n→ The nonlinear bend between them prevents the collapse.")
print("  That bend is what makes stacking layers worthwhile.")


## Attention Is a Matrix Multiply Too

The Attention chapter had the word "it" looking back through a sentence for a noun to attach to.
Scoring every candidate — **cat**, **dog**, and filler **blah** — against the query for "it"
is just a detector stack: stack the tokens' key-vectors as rows, multiply by the query, and out come the match scores.

Here are the 2-dimensional key vectors from the chapter's worked example:

In [ ]:
# 2-dim key vectors for each token
#   cat  → [1, 0]  (noun)
#   dog  → [1, 0]  (noun)
#   blah → [0, 1]  (filler)
keys = torch.tensor([
    [1.0, 0.0],   # cat
    [1.0, 0.0],   # dog
    [0.0, 1.0],   # blah
], dtype=torch.float32)

token_names = ['cat', 'dog', 'blah']

# Query vector for "it": [1, 0]  (looks for nouns)
query = torch.tensor([1.0, 0.0])

# One matrix-vector multiply gives all match scores at once
match_scores = keys @ query

print("Match scores (Keys @ query):")
for name, score in zip(token_names, match_scores):
    print(f"  {name:4s}: {score:.1f}")
print("\nNouns score 1.0, filler scores 0.0 — exactly as in the chapter.")
print("Same operation as the detector stack: keys are detectors, query is the input.")


The query, key, and value vectors used in real attention are themselves produced by matrix multiplies applied to each token's embedding. Blending the chosen values together is one more matrix multiply. Attention is matrix multiplication the whole way down. (Full details in the Attention chapter.)

## The Same Math Spins 3D Worlds

As a bonus: the exact same multiply-and-add that re-describes an animal also rotates a point in space. Every frame of a 3D game applies rotation matrices to every vertex. GPUs were built to do millions of these per frame — and then AI turned out to be mostly matrix multiplication too. The same chip that spins dragons runs transformer layers.

In [ ]:
import math

# Rotate the point (1, 0) by 45 degrees
theta = math.pi / 4
rotation = torch.tensor([
    [math.cos(theta), -math.sin(theta)],
    [math.sin(theta),  math.cos(theta)],
])

point = torch.tensor([1.0, 0.0])
rotated = rotation @ point

print(f"Original point: {point.tolist()}")
print(f"Rotated 45°:    [{rotated[0]:.3f}, {rotated[1]:.3f}]")
print("(Should be ≈ [0.707, 0.707] — moved 45° around the origin)")


---

*This notebook accompanies [Chapter 10: A Thousand Questions at Once](https://learnai.robennals.org/matrices). The interactive widgets in the web version let you explore these concepts visually.*

*New to PyTorch? See the [PyTorch from Scratch](https://learnai.robennals.org/appendix-pytorch) appendix for a beginner-friendly introduction.*